# PNCP — Identificação de Subenquadramento de Engenharia

TCC MBA IA & Big Data (ICMC/USP) — Profa. Solange O. Rezende

## Metodologia em 3 etapas

1. **Triagem determinística (etapa 0)** — pré-filtro lexical + verificação
   de rito de engenharia (ART/RRT, memorial, NBR…). Casos óbvios são
   reclassificados aqui, antes do ML.
2. **ML para os ambíguos** — TF-IDF + LR/RF/SVC, opcional BERTimbau.
3. **Camadas complementares** — outliers, grafos, CNAE, aditivos.

**Como usar:** rode em ordem. Cada etapa **persiste em disco** (parquet/json).
Se o kernel cair, é só continuar da célula seguinte — nada se perde.

**Atualizar código no meio:** se eu fizer commits novos enquanto você roda,
execute `pncp.atualizar()` numa célula vazia, depois `import pncp` de novo.

## Célula 1 — Setup (rode 1× por sessão)

In [ ]:
!pip install -q wordcloud tqdm imbalanced-learn nltk mlxtend psutil
!pip install -q pymupdf pdfplumber pytesseract networkx python-louvain openpyxl
!pip install -q sentence-transformers transformers torch statsmodels
!apt-get install -y -qq tesseract-ocr tesseract-ocr-por > /dev/null
print('OK — dependências instaladas')

## Célula 2 — Drive + clonar o repositório

A cada execução, força `git fetch + reset --hard` para garantir que está
rodando a última versão da branch `refactor`. Você verá o commit ativo
impresso para conferência.

In [ ]:
import sys, os, subprocess

REPO_URL = 'https://github.com/caiofvserra/pncp-engineering-analytics.git'
REPO_DIR = '/content/pncp-engineering-analytics'
BRANCH = 'refactor'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=False)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=False)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard',
                    f'origin/{BRANCH}'], check=True)

# Confirmação visual: você verá o último commit (deve ser o mais recente)
print(subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--oneline'],
                     capture_output=True, text=True).stdout)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
for mod in [m for m in list(sys.modules) if m.startswith('pncp')]:
    del sys.modules[mod]

import pncp
pncp.montar_drive('/content/drive/MyDrive/PNCP_TCC')
pncp.keep_alive()
pncp.ram.monitorar_ram('início')

## Célula 3 — Coleta (com retomada automática)

Endpoint estável `/v1/contratos`, **mês a mês**, com 6 retentativas e
backoff exponencial (4-64s). A cada 10 páginas grava checkpoint
(parquet + JSON de progresso).

**Se o Colab desconectar:** reabra o notebook, rode as células 1-2 e
depois esta de novo. A função detecta meses já completos e a última
página parcial e **retoma do ponto exato**, sem rebaixar nada.

Para inspecionar progresso sem baixar: `pncp.coleta.status(uf='SP', anos=range(2024, 2027))`

In [ ]:
# Coleta SP de 2024 a 2026. Retoma automaticamente parciais
# e meses pendentes. Pula meses já completos.
pncp.coleta.coletar(
    uf='SP',
    anos=range(2024, 2027),
    mes_inicio=1, mes_fim=12,
    max_paginas=1000,
    tamanho=500,
)

# SEMPRE regenera o consolidado contratos.parquet com os parciais
# atuais. Isto garante que Cell 4+ veja os anos certos mesmo se
# a sessão anterior coletou mais dados depois.
pncp.coleta.regenerar_consolidado(uf='SP')

# Diagnóstico final
pncp.coleta.diagnosticar(uf='SP')
pncp.ram.monitorar_ram('após coleta')


### Recuperação (opcional)

Use se você baixou em sessões separadas (ex: 1 ano por vez) e quer
consolidar tudo num único parquet, sem chamar a API.

In [ ]:
# Inspeciona estado da coleta (sem baixar nada):
# pncp.coleta.status(uf='SP', anos=range(2024, 2027))

# Junta vários parquets de coletas separadas num só:
# pncp.coleta.combinar_parquets(uf='SP')

## Célula 4 — EDA + texto + TF-IDF (etapa estritamente local)

Gera ~12 gráficos descritivos (distribuição, série temporal, top
municípios, top órgãos, comprimento, palavras, bigramas, esfera,
estabilidade temporal) e a matriz TF-IDF para o classificador.

NÃO baixa PDFs nem chama APIs — só processa `contratos.parquet`.
Se já rodou, a mensagem "TF-IDF já existe" significa cache OK
(refaça com `pncp.texto.construir_tfidf(caminho, forcar=True)`).

O gráfico de **marcadores por rótulo** (ART/RRT/CREA) só aparece
DEPOIS da Célula 6 (PDFs) — é gerado automaticamente lá.

In [ ]:
from pncp import config

caminho = config.caminho(config.SUB_COLETA, 'contratos.parquet')

pncp.eda.executar(caminho_parquet=caminho)
pncp.texto.preprocessar(caminho)
pncp.texto.marcar_termos_dominio(caminho)
pncp.texto.construir_tfidf(caminho)
pncp.ram.monitorar_ram('após texto/TF-IDF')

## Célula 5 — Triagem inicial (pré-filtro lexical)

Identifica contratos 'geral' que são *obviamente* engenharia pelo objeto.
Sem os PDFs ainda, todos viram `subenquadramento_real` provisório — vão
ser reclassificados na Célula 7 quando tivermos as features de rito.

In [ ]:
pncp.triagem.executar()

## Célula 6 — PDFs (Camada 2): TR/Edital dos óbvios + ambíguos

Prioriza os contratos marcados como `obvio_engenharia` na triagem para
saber se seguiram o rito.

In [ ]:
pncp.pdfs.executar(max_contratos=300)
pncp.ram.monitorar_ram('após PDFs')

## Célula 7 — Triagem refinada (com verificação de rito)

Re-roda a triagem cruzando com features dos PDFs:
- `rotulacao_incorreta_processo_ok` = óbvio + rito seguido (≥2 sinais)
- `subenquadramento_real` = óbvio + rito **não** seguido (violação Lei 14.133)
- `ambiguo` = vai para o ML

In [ ]:
pncp.triagem.executar()

## Célula 8 — Classificação supervisionada (ML para os ambíguos)

In [ ]:
# Validação temporal: treina nos anos passados, testa em 2026.
# Use isso para mostrar que o modelo generaliza para dados futuros.
pncp.classificacao.executar(
    fazer_grid=True,
    fazer_holdout=True,
    fazer_mcnemar=True,
    fazer_bootstrap=True,
    # holdout_anos=[2026],   # descomente p/ treinar em 2024-25 e testar em 2026
)
pncp.ram.monitorar_ram("após classificação")

## Célula 9 — Detecção de outliers (anomalias contextuais)

IsolationForest + One-Class SVM + Z-score do valor + ensemble.

In [ ]:
pncp.outliers.executar(
    fazer_iforest=True,
    fazer_ocsvm=True,
    fazer_zscore=True,
    fazer_ensemble=True,
    fazer_lof=False,
)

## Célula 10 — Técnicas avançadas (LDA, Label Propagation, Apriori, KMeans, GMM)

In [ ]:
pncp.avancado.executar(
    fazer_lda=True,
    fazer_lp=True,
    fazer_apriori=True,
    fazer_kmeans=True,
    fazer_hier=True,
    fazer_gmm=True,
)
pncp.ram.monitorar_ram('após avançado')

## Célula 11 — Embeddings BERTimbau (opcional, mais lento)

Use `tipo='sentence-bert'` para iterar rápido; `'bertimbau'` na versão final.

In [ ]:
# pncp.embeddings.executar(tipo='sentence-bert', treinar=True)
# pncp.ram.monitorar_ram('após embeddings')

## Célula 11.5 — Validação semântica via LLM local (opcional)

Llama 3.1 (via Ollama) lê cada suspeito top e dá veredicto textual
com justificativa, indicação se exige ART/RRT e recomendação.
Roda local, gratuito.

In [ ]:
# Setup 1x por sessão (~5min para baixar 5GB do Llama 3.1):
# !curl -fsSL https://ollama.com/install.sh | sh

# pncp.llm.iniciar_ollama()
# pncp.llm.pull_modelo("llama3.1")
# pncp.llm.validar_suspeitos(top_n=20)
# pncp.llm.mostrar(top_n=10)

## Célula 12 — Aditivos + Grafos + CNAE

Cada camada num bloco isolado: se uma falhar (rate limit, dataset pequeno,
rede ruim) as outras continuam.

In [ ]:
import traceback

def _tentar(nome, fn, *args, **kwargs):
    try:
        fn(*args, **kwargs)
        print(f'  ✅ {nome}')
    except Exception as e:
        print(f'  ⚠ {nome} falhou: {type(e).__name__}: {e}')
        traceback.print_exc(limit=2)

_tentar('aditivos', pncp.aditivos.executar, max_contratos=200)
_tentar('grafos',   pncp.grafos.executar)
_tentar('cnae',     pncp.cnae.executar,
        max_consultas=200,
        caminho_excel_crea='/content/drive/MyDrive/PNCP_TCC/cnaes_crea.xlsx')

pncp.ram.monitorar_ram('após complementares')

## Célula 13 — Relatório final TCC + consolidação

Combina o veredito da triagem (determinístico) com o ranking ML.

In [ ]:
pncp.relatorio.gerar()
from pathlib import Path
rel = Path(pncp.config.PASTA_DADOS) / pncp.config.SUB_P9 / 'relatorio.md'
print(rel.read_text(encoding='utf-8'))

## Célula 14 — Validação contra ground truth manual (opcional)

1. Abra `dados/cnae/amostra_revisao_manual.csv`
2. Preencha a coluna `revisao_manual`: `subenq` / `ok` / `duv`
3. Rode esta célula

In [ ]:
# Esta célula é OPT-IN. Em "Run all" só roda se o CSV já estiver
# preenchido manualmente. Caso contrário, pula com aviso amigável.
import pandas as pd
from pathlib import Path

csv = pncp.config.caminho(pncp.config.SUB_P8, 'amostra_revisao_manual.csv')

if not Path(csv).exists():
    print(f"⏭ pulando — CSV ainda não existe: {csv}")
elif 'revisao_manual' not in pd.read_csv(csv, nrows=1).columns:
    print(f"⏭ pulando — coluna 'revisao_manual' ausente em {csv.name}")
else:
    n = pd.read_csv(csv)['revisao_manual'].dropna().astype(str).ne('').sum()
    if n == 0:
        print(f"⏭ pulando — nenhuma linha preenchida em {csv.name}")
        print(f"   Para validar: abra o CSV, preencha 'revisao_manual' com")
        print(f"   subenq/ok/duv linha a linha, salve, e rode esta célula.")
    else:
        print(f"✓ {n} linhas revisadas — validando")
        pncp.relatorio.validar_ground_truth(csv)


## Análises extras (notebooks do MBA)

### Decomposição temporal + Change Points
Notebook 03 (Decomposição) + 11 (Outras) do curso. Mostra tendência,
sazonalidade e quebras estruturais nos contratos 'geral'.

In [ ]:
pncp.temporal.decompor_serie(rotulo='geral')
pncp.temporal.detectar_change_points(rotulo='geral')
# pncp.temporal.forecast_volume(rotulo='geral', n_meses=12)

### RAG — busca de contratos similares aos suspeitos
Notebook 08 (Embeddings) + Tutoria 13_05 (RAG). Dado um suspeito
confirmado, encontra outros parecidos. Útil pra investigação jurídica.

In [ ]:
# Pré-requisito: pncp.embeddings.gerar() rodado (gera dados/embeddings)
# pncp.similaridade.expandir_suspeitos(top_n=20, k_por_suspeito=10)

# Busca livre por texto:
# pncp.similaridade.buscar_por_texto(
#     "manutenção elétrica predial com substituição de quadros e ART do CREA"
# )

### Mapa geográfico + NER
Notebook 14 (Informação Geográfica). NER via Spacy + HeatMap Folium dos
municípios com mais suspeitos. Salva HTML interativo.

In [ ]:
# Pré-requisitos no Colab:
# !pip install -q spacy geopy folium
# !python -m spacy download pt_core_news_sm

# pncp.geografico.extrair_entidades(amostra=300)
# pncp.geografico.geocodificar_municipios(amostra=100)
# pncp.geografico.mapa_suspeitos()

# Alternativa LLM (extração estruturada):
# pncp.llm.extrair_entidades_llm(top_n=30)
# pncp.llm.resumir_objetos(amostra=20)

### Grafo semântico de contratos + propagação de rótulos

Notebook 09 P3 + 12 P2 + 18 (Agentes) do curso. Constrói rede k-NN
sobre embeddings dos contratos, detecta comunidades por label
propagation, e propaga rótulos a partir de seeds confirmados.

Diferente do `pncp.grafos` que é bipartido órgão↔fornecedor —
este conecta contratos por SIMILARIDADE SEMÂNTICA.

In [ ]:
# Pré-requisito: pncp.embeddings.gerar() rodado

# pncp.grafos_semanticos.construir(k=5, focar_em='geral')

# Depois de revisar manualmente alguns suspeitos, propaga via cluster:
# seeds = ['12345678000199-1-000123/2024', ...]
# pncp.grafos_semanticos.propagar_rotulos(seeds)

### Indicadores gerados por LLM a partir de clusters

Notebook 12 P2. Para cada cluster grande de contratos similares,
o Llama 3.1 gera UM indicador JSON estruturado:
`{nome, categoria, descricao, indicio_subenquadramento, recomendacao_acao}`.

In [ ]:
# pncp.llm.gerar_indicadores(top_n_clusters=10, n_por_cluster=5)
# Saída em dados/llm/indicadores.json

### Agente LLM com ferramentas (perguntas em linguagem natural)

Notebook 18 do curso. Agente Llama 3.1 com 4 ferramentas:
buscar_similares, listar_suspeitos, contar_por_rotulo, buscar_municipio.
Você pergunta em português, o agente escolhe a ferramenta e responde.

In [ ]:
# resposta = pncp.llm.agente(
#     "Quais os 5 contratos mais parecidos com manutenção elétrica predial?"
# )
# print(resposta)

## Snapshot de resultados (recomendado)

Salva uma cópia dos resultados em `dados/snapshots/resultados_<data>/`.
NÃO copia coleta nem cache de PDFs (gigantes e não mudam).

A `pncp.relatorio.gerar()` já chama isso automaticamente, mas
você pode forçar quando quiser (ex: antes de mudar parâmetros
e re-rodar):

In [ ]:
# Cria pasta dados/snapshots/resultados_<data_hora>/ com tudo
# exceto coleta e cache PDF (não copia gigabytes desnecessários)
pncp.snapshot_resultados()

## Glossário

In [ ]:
pncp.relatorio.glossario()